# Analyse des outputs de MR-CLIP

In [1]:
import pickle

path = "/NAS/coolio/benolive/Diffusion_beta_encoder_2D/logs_mrclip/inference/mr_clip_2d_et20_rt20/dataset_train_rgb224/checkpoints/image_embeddings.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

print(data.keys())
print(data["embeddings"].shape)
print(len(data["filepaths"]))
print(data["filepaths"][:5])

dict_keys(['embeddings', 'labels', 'filepaths'])
(1000, 512)
1000
['/NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/aibl_1055820.png', '/NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/aibl_1055836.png', '/NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/aibl_1078793.png', '/NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/aibl_1081897.png', '/NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/aibl_1081962.png']


In [4]:
import numpy as np

emb = np.asarray(data["embeddings"], dtype=np.float32)
filepaths = data["filepaths"]

print("Shape embeddings:", emb.shape)
print("Nombre de filepaths:", len(filepaths))

Shape embeddings: (1000, 512)
Nombre de filepaths: 1000


In [5]:
# ------------------------------------------------------------
# 1) Vérifier les embeddings exactement identiques
# ------------------------------------------------------------
unique_exact = np.unique(emb, axis=0)

print("\n=== Identité exacte ===")
print("Nombre d'embeddings:", len(emb))
print("Nombre d'embeddings uniques exacts:", len(unique_exact))
print("Nombre de doublons exacts:", len(emb) - len(unique_exact))



=== Identité exacte ===
Nombre d'embeddings: 1000
Nombre d'embeddings uniques exacts: 1000
Nombre de doublons exacts: 0


In [6]:
# ------------------------------------------------------------
# 2) Normalisation L2 pour calculer des similarités cosinus
# ------------------------------------------------------------
norms = np.linalg.norm(emb, axis=1, keepdims=True)
emb_norm = emb / np.clip(norms, 1e-12, None)

# Matrice de similarité cosinus : shape (N, N)
sim = emb_norm @ emb_norm.T

# On retire la diagonale, car chaque embedding est identique à lui-même
n = sim.shape[0]
mask = ~np.eye(n, dtype=bool)
sim_off_diag = sim[mask]

print("\n=== Similarités cosinus hors diagonale ===")
print("Min :", sim_off_diag.min())
print("Mean:", sim_off_diag.mean())
print("Std :", sim_off_diag.std())
print("Max :", sim_off_diag.max())


=== Similarités cosinus hors diagonale ===
Min : 0.95612216
Mean: 0.9960609
Std : 0.003478774
Max : 0.9999963


In [7]:
# ------------------------------------------------------------
# 3) Compter les paires presque identiques
# ------------------------------------------------------------
thresholds = [0.999999, 0.99999, 0.9999, 0.999, 0.99, 0.95]

print("\n=== Nombre de paires très similaires ===")
for th in thresholds:
    count = np.sum(sim_off_diag > th)
    # Chaque paire apparaît deux fois dans sim : (i,j) et (j,i)
    count_unique_pairs = count // 2
    print(f"cosine > {th}: {count_unique_pairs} paires uniques")


=== Nombre de paires très similaires ===
cosine > 0.999999: 0 paires uniques
cosine > 0.99999: 1 paires uniques
cosine > 0.9999: 70 paires uniques
cosine > 0.999: 61324 paires uniques
cosine > 0.99: 467134 paires uniques
cosine > 0.95: 499500 paires uniques


Il n'y a donc pas de duplication parfaite énorme car sur des très grande similarité, très peu de pairs avec même embedding. En revanche, les pairs sont globalement très proches.

Les questions sont : 
- Est-ce normal d'avoir des embeddings très similaires ou selon le dataset, les embeddings devraient être très variés ?
- J'ai normalisé chaque slice indépendamment, est-ce que cela a un grand impact ?

Etudions alors les pairs les moins similaires :

In [8]:
# Paires les moins similaires
sim_no_diag = sim.copy()
np.fill_diagonal(sim_no_diag, np.inf)

bottom_k = 10
flat_indices = np.argpartition(sim_no_diag.ravel(), bottom_k)[:bottom_k]
pairs = np.array(np.unravel_index(flat_indices, sim_no_diag.shape)).T
pairs = sorted(pairs, key=lambda ij: sim_no_diag[ij[0], ij[1]])

seen = set()

print("\n=== Top paires les moins similaires ===")
shown = 0
for i, j in pairs:
    if i == j:
        continue

    pair_key = tuple(sorted((int(i), int(j))))
    if pair_key in seen:
        continue
    seen.add(pair_key)

    print(f"\nSimilarité cosinus: {sim_no_diag[i, j]:.8f}")
    print(f"  [{i}] {filepaths[i]}")
    print(f"  [{j}] {filepaths[j]}")

    shown += 1
    if shown >= 5:
        break


=== Top paires les moins similaires ===

Similarité cosinus: 0.95612216
  [139] /NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/hcp_178950R2.png
  [755] /NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/oas-bio_455Sd2887R1.png

Similarité cosinus: 0.95777547
  [755] /NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/oas-bio_455Sd2887R1.png
  [246] /NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/icbm-ACS_2992.png

Similarité cosinus: 0.95811087
  [755] /NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/oas-bio_455Sd2887R1.png
  [336] /NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/icbm-sonata_17994.png

Similarité cosinus: 0.95912582
  [337] /NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/icbm-sonata_17995.png
  [755] /NAS/coolio/benolive/Diffusion_beta_encoder_2D/data/brain_slices/train/png/oas-bio_455Sd288